In [2]:
# !pip install scanpy anndata scikit-learn joblib matplotlib scipy numpy pandas h5py 

In [5]:
import os
import numpy as np 
import pandas as pd 
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse

In [ ]:
# ZENODO = "https://zenodo.org/records/17888140/files"

In [ ]:
# download(f"{ZENODO}/combine_training.h5ad", "data/combine_training.h5ad")

  done (3,793,572,854 bytes)


In [ ]:
# download(f"{ZENODO}/combined_tumor_up_down_degs.txt", "data/combined_tumor_up_down_degs.txt")

  done (16,830 bytes)


In [ ]:
# download(f"{ZENODO}/test_cancerCellLine.h5ad", "data/test_cancerCellLine.h5ad")

  done (2,294,934,772 bytes)


In [1]:
# download(f"{ZENODO}/test_TabulaSapiens.h5ad", "data/test_TabulaSapiens.h5ad")

In [6]:
train_adata = sc.read_h5ad("data/combine_training.h5ad")

In [7]:
train_adata

AnnData object with n_obs × n_vars = 416774 × 2707
    obs: 'Raw_annotation'

In [7]:
train_adata.obs.head()

,Raw_annotation
GTCCTCAGTGGCAAAC-1-HCATisStab7747198-11,Normal
CGATCGGTCGAACTGT-1_7-6,Normal
GCTTGAAAGTACGTAA-1-25-8,Tumor
ACATACGGTACCGTTA_LUNG_N18-5,Normal
19_GAAGTTAGATGA-20,Normal


In [9]:
train_adata.obs["Raw_annotation"].value_counts()

Raw_annotation
Normal    282721
Tumor     134053
Name: count, dtype: int64

In [8]:
feature_list = list(np.loadtxt("data/combined_tumor_up_down_degs.txt", dtype=str))

In [11]:
feature_list[:20]

[np.str_('EMC4'),
 np.str_('EIF5'),
 np.str_('CDA'),
 np.str_('NDUFS8'),
 np.str_('DYNC1H1'),
 np.str_('LSM4'),
 np.str_('DDX27'),
 np.str_('FKBP4'),
 np.str_('SLC52A2'),
 np.str_('TPM3'),
 np.str_('PSMA2'),
 np.str_('NOLC1'),
 np.str_('CDKN2A'),
 np.str_('HRAS'),
 np.str_('DUT'),
 np.str_('ETV4'),
 np.str_('EIF4A3'),
 np.str_('CYP4F3'),
 np.str_('C11orf58'),
 np.str_('SLC25A3')]

In [19]:
from sklearn.preprocessing import MinMaxScaler

In [13]:
train_adata.var_names_make_unique()

In [9]:
sc.pp.normalize_total(train_adata, target_sum=1e4)

In [10]:
cancer_test = sc.read_h5ad("data/test_cancerCellLine.h5ad")
cancer_test.var_names_make_unique()
sc.pp.normalize_total(cancer_test, target_sum=1e4)

In [17]:
cancer_test

AnnData object with n_obs × n_vars = 53513 × 26989
    obs: 'Dataset', 'Test_label', 'Raw_annotation', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Angiogenesis', 'Apoptosis', 'Cell_Cycle', 'DNA_damage', 'DNA_repair', 'Differentiation', 'EMT', 'Hypoxia', 'Inflammation', 'Invasion', 'Metastasis', 'Proliferation', 'Quiescence', 'Stemness', 'Tumor', 'Sustaining Proliferative Signaling', 'Evading Growth Suppressors', 'Avoiding Immune Destruction', 'Enabling Replicative Immortality', 'Tumour-Promoting Inflammation', 'Activating Invasion and Metastasis', 'Inducing Angiogenesis', 'Genome Instability and Mutation', 'Resisting Cell Death', 'Deregulating Cellular Energetics', 'Basal-like (S01)', 'Normal-enriched (S02)', 'Pro-angiogenic (S03)', 'Pro-inflammatory (S04)', 'Metabolic (S06)'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'

In [11]:
train_genes = set(train_adata.var_names)
cancer_genes = set(cancer_test.var_names)

In [12]:
normal_test = sc.read_h5ad("data/test_TabulaSapiens.h5ad")
normal_test.var_names_make_unique()
sc.pp.normalize_total(normal_test, target_sum=1e4)

In [13]:
normal_genes = set(normal_test.var_names)

In [14]:
normal_features = sorted(set(feature_list) & train_genes & normal_genes)

In [15]:
label_map = {"Normal": 0, "Malignant": 1, "Tumor": 1}
y_train = train_adata.obs["Raw_annotation"].map(label_map).values

In [38]:
f"Normal: {(y_train == 0).sum():,}"

'Normal: 282,721'

In [39]:
f"Malignant: {(y_train == 1).sum():,}"

'Malignant: 134,053'

In [40]:
X_train_cancer = train_adata[:, cancer_features].X
if hasattr(X_train_cancer, 'toarray'):
    X_train_cancer = X_train_cancer.toarray()

In [42]:
y_train = np.asarray(y_train)

In [20]:
from sklearn.linear_model import LogisticRegression

In [44]:
lr_model = LogisticRegression(n_jobs=1, max_iter=1000)
lr_model.fit(X_train_cancer, y_train)

/home/pandeyps/.pyenv/versions/3.11.9/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [45]:
# prediction on cancer cells

In [46]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [16]:
cancer_features = sorted(set(feature_list) & train_genes & cancer_genes)

In [47]:
X_cancer = cancer_test[:, cancer_features].X
if hasattr(X_cancer, 'toarray'):
    X_cancer = X_cancer.toarray()

In [48]:
y_pred_cancer = lr_model.predict(X_cancer)
y_prob_cancer = lr_model.predict_proba(X_cancer)[:, 1]

In [49]:
cancer_test.obs["prediction"] = pd.Categorical(
    ["Malignant" if p else "Normal" for p in y_pred_cancer])
cancer_test.obs["malignancy_probability"] = y_prob_cancer

In [50]:
cancer_test.obs["prediction"].value_counts()

prediction
Malignant    53500
Normal          13
Name: count, dtype: int64

In [51]:
cancer_test.obs[["prediction", "malignancy_probability"]].head(10)

,prediction,malignancy_probability
TCTCTAATCCGTTGTC-5-6,Malignant,0.972683
ATGGGAGAGGGAAACA-4-22,Malignant,0.962168
CGCGTTTCAAGAAGAG-7-16,Malignant,0.968120
TGACTTTTCTGGAGCC-3-22,Malignant,0.985432
ACCGTAAAGACTAGGC-6-6,Malignant,0.962695
c3590,Malignant,0.952022
GCATGCGCAACTGCGC-14-15,Malignant,0.990817
TGGTTAGTCACTATTC-3-22,Malignant,0.972221
CCACGGACACTCAGGC-14-15,Malignant,0.941239
GCGCAGTTCGGCCGAT-10-9,Malignant,0.937519


In [52]:
# prediction on normal cells

In [17]:
X_train_normal = train_adata[:, normal_features].X

In [21]:
lr_model_n = LogisticRegression(n_jobs=1, max_iter=1000)
lr_model_n.fit(X_train_normal, y_train)

/home/pandeyps/.pyenv/versions/3.11.9/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [22]:
X_normal = normal_test[:, normal_features].X

In [23]:
y_pred_normal = lr_model_n.predict(X_normal)
y_prob_normal = lr_model_n.predict_proba(X_normal)[:, 1]

In [24]:
normal_test.obs["prediction"] = pd.Categorical(
    ["Malignant" if p else "Normal" for p in y_pred_normal])
normal_test.obs["malignancy_probability"] = y_prob_normal

In [25]:
normal_test.obs["prediction"].value_counts()

prediction
Normal       80691
Malignant    23457
Name: count, dtype: int64